# LOAD DATA

In [ ]:
import pandas as pd
import numpy as np
df_deep_train=pd.read_csv("/kaggle/input/multiclass-deep-features/train_features_multiclass.csv")
df_deep_test=pd.read_csv("/kaggle/input/multiclass-deep-features/test_features_multiclass.csv")
df_handcrafted_train=pd.read_csv("/kaggle/input/multiclass-handcrafted-features/Hnadcrafted_Train_features_multiclass.csv")
df_handcrafted_test=pd.read_csv("/kaggle/input/multiclass-handcrafted-features/Handcraftred_Test_features_multiclass.csv")

In [ ]:
df_deep_train['id']=df_deep_train['image_id']
df_deep_test['id']=df_deep_test['image_id']

In [ ]:
#get fusion features
df_fusion_train=pd.merge(df_deep_train,df_handcrafted_train,on='id',how='inner')
df_fusion_test=pd.merge(df_deep_test,df_handcrafted_test,on='id',how='inner')

#remove reduntant and improper data
X_fusion_train=df_fusion_train.drop(columns=['image_id','id','label_y','label_x'])
X_fusion_test=df_fusion_test.drop(columns=['image_id','id','label_y','label_x'])

#classes
y_train=df_fusion_train['label_y']
y_test=df_fusion_test['label_y']

#concatenate
X_total = np.concatenate((X_fusion_train, X_fusion_test), axis=0)
y_total = np.concatenate((y_train, y_test), axis=0)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_total)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt

# Use your processed data
X = X_scaled   # features after scaling
y = y_total          # labels


# SCREE PLOT

In [ ]:

# 1. PCA and Scree Plot
pca = PCA()
pca.fit(X)
explained_var = pca.explained_variance_ratio_
print("Scree plot")
plt.figure(figsize=(8,5))
plt.plot(np.arange(1, len(explained_var) + 1), explained_var, marker='o')
plt.xlabel("Number of PCA Components")
plt.ylabel("Explained Variance Ratio")
plt.title("Scree Plot: Explained Variance per Component")
plt.grid()
plt.show()


# Cumulative Explained Variance Table

In [ ]:


cum_var = np.cumsum(explained_var)
table = []
for n in [1, 5, 10, 15, 20, 50, 100, 150, 200, 250, 300, 350,400, 500, 600, 700, 800, 900, 1000, len(explained_var)]:
    if n <= len(cum_var):
        table.append([n, cum_var[n-1]])
cum_table = pd.DataFrame(table, columns=['Components', 'Cumulative Variance'])
print(cum_table)

# Pick n so that cum_var[n-1] >= 0.9 (i.e., 90% variance)

# PCA PERFORMANCE PLOT

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import numpy as np
import matplotlib.pyplot as plt


n_splits = 5
train_size_per_fold = int(len(X_total) * (1 - 1/n_splits))
max_comp = min(train_size_per_fold, X_total.shape[1]) 

print(f"Total Samples: {len(X_total)}")
print(f"Max allowable components for {n_splits}-fold CV: {max_comp}")

scores = []
stds = []
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)


step_size = 5 
# Ensure the range does not exceed the max_comp limit
component_range = list(range(1, max_comp, step_size))

print(f"Starting PCA Grid Search (Step={step_size})...")

for n_comp in component_range:
    pipe = Pipeline([
        ('scaler', StandardScaler()),  
        ('pca', PCA(n_components=n_comp, random_state=42)),
        ('clf', LogisticRegression(max_iter=10000, random_state=42, n_jobs=-1))
    ])
    
    cv_scores = cross_val_score(pipe, X_total, y_total, cv=skf, scoring='accuracy', n_jobs=-1)
    
    scores.append(np.mean(cv_scores))
    stds.append(np.std(cv_scores))
    print(f"n={n_comp}: Acc={np.mean(cv_scores):.4f}")


best_idx = np.argmax(scores)
best_n = component_range[best_idx]
print(f"\n--- Result ---")
print(f"Best Accuracy: {scores[best_idx]:.4f} with {best_n} components")

plt.figure(figsize=(10, 6))
plt.errorbar(component_range, scores, yerr=stds, fmt='-o', color='navy', 
             capsize=3, ecolor='lightgray', elinewidth=1, label='CV Accuracy')
plt.axvline(x=best_n, color='red', linestyle='--', label=f'Best n={best_n}')
plt.xlabel('Number of PCA Components')
plt.ylabel('Cross-Validation Accuracy')
plt.title('Optimization: Logistic Regression Accuracy vs. PCA Components')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()